In [7]:
import numpy as np
import torch

class ConnectFourEnv():
    def __init__(self) -> None:
        self.rows = 6
        self.cols = 7
        # 1D board of size 42
        self.board = np.zeros(shape=(self.rows * self.cols,), dtype=int)
        self.computer = 1
        self.opponent = -1
        self.reset()

    def reset(self):
        """Resets the board and handles who moves first."""
        self.board[:] = 0
        self.done = False
        self.winner = None
        
        # Randomly decide who goes first
        self.mover = np.random.choice([self.computer, self.opponent])
        
        # If opponent starts, they make a random move immediately
        if self.mover == self.opponent:
            action = self.random_action()
            self.apply_action(action, self.opponent)
            
        return self.board.copy()

    def available_actions_idx(self):
        """Returns a list of column indices (0-6) that are not full."""
        # Reshape to 2D to easily check the top row (row 0)
        board_2d = self.board.reshape(self.rows, self.cols)
        # If the top row (0) at column c is 0, the column is valid
        return [c for c in range(self.cols) if board_2d[0, c] == 0]

    def random_action(self):
        """Returns a random valid column."""
        possible_cols = self.available_actions_idx()
        if not possible_cols:
            return None # Draw/Full
        return np.random.choice(possible_cols)

    def apply_action(self, col_idx, player):
        """
        Simulates gravity: places the player's piece in the 
        lowest available row in the given column.
        """
        board_2d = self.board.reshape(self.rows, self.cols)
        
        # Find the lowest empty row in this column
        # We scan from bottom (row 5) to top (row 0)
        for r in range(self.rows - 1, -1, -1):
            if board_2d[r, col_idx] == 0:
                board_2d[r, col_idx] = player
                break
        
        # Flatten back to 1D to update self.board
        self.board = board_2d.flatten()

    def check_win(self, player):
        """Checks horizontal, vertical, and diagonal lines for 4 connected."""
        board_2d = self.board.reshape(self.rows, self.cols)

        # 1. Horizontal
        for r in range(self.rows):
            for c in range(self.cols - 3):
                if np.all(board_2d[r, c:c+4] == player):
                    return True

        # 2. Vertical
        for r in range(self.rows - 3):
            for c in range(self.cols):
                if np.all(board_2d[r:r+4, c] == player):
                    return True

        # 3. Diagonal (\)
        for r in range(self.rows - 3):
            for c in range(self.cols - 3):
                if np.all([board_2d[r+i, c+i] == player for i in range(4)]):
                    return True

        # 4. Anti-Diagonal (/)
        for r in range(3, self.rows):
            for c in range(self.cols - 3):
                if np.all([board_2d[r-i, c+i] == player for i in range(4)]):
                    return True

        return False
    
    def step(self, action, opponent_model=None): # <--- Check this argument
        # 1. Check Agent Valid Move
        if action not in self.available_actions_idx():
             return self.board.copy(), -10, True, {"result": "Error"}
        
        # 2. Agent Move
        self.apply_action(action, self.computer)
        if self.check_win(self.computer):
            return self.board.copy(), 1, True, {"result": "Win"}
        if len(self.available_actions_idx()) == 0:
            return self.board.copy(), 0, True, {"result": "Draw"}

        # 3. Opponent Move
        if opponent_model is None:
            # Default: Random
            opp_action = self.random_action()
        else:
            # Advanced: The Clone
            opp_action = self.get_opponent_action(opponent_model) # <--- Make sure this is called
            
        self.apply_action(opp_action, self.opponent)

        if self.check_win(self.opponent):
            return self.board.copy(), -1, True, {"result": "Loss"}
        if len(self.available_actions_idx()) == 0:
            return self.board.copy(), 0, True, {"result": "Draw"}

        return self.board.copy(), 0, False, {}

    def get_opponent_action(self, model):
        # 1. Prepare the board (Flip perspective)
        board_for_opp = self.board * -1 
        
        # 2. Create the tensor (defaults to CPU)
        state_t = torch.tensor(board_for_opp, dtype=torch.float32).unsqueeze(0).view(1, 1, 6, 7)
        
        # 3. CRITICAL FIX: Move tensor to the same device as the model (CPU or GPU)
        # We check the device of the first parameter of the model
        device = next(model.parameters()).device
        state_t = state_t.to(device)
        
        # 4. Get the move
        with torch.no_grad():
            q_vals = model(state_t)
            valid_moves = self.available_actions_idx()
            
            # Mask invalid moves
            mask = torch.full_like(q_vals, -float('inf'))
            mask[0, valid_moves] = q_vals[0, valid_moves]
            
            action = mask.max(1)[1].item()
            
        return action

    def render(self):
        """Visualizes the board."""
        board_2d = self.board.reshape(self.rows, self.cols)
        symbols = {0: '.', 1: 'X', -1: 'O'}
        print("\nBoard State:")
        for row in board_2d:
            print(" ".join([symbols[x] for x in row]))
        print("-" * 13)
        print("0 1 2 3 4 5 6\n")

In [8]:
import torch.nn as nn
import torch.nn.functional as F

class QNConnectFour(nn.Module):
    def __init__(self, output_dim=7):
        super(QNConnectFour, self).__init__()
        
        # --- Convolutional Block ---
        # We treat the board as an image: 1 channel (the values -1, 0, 1), 6 rows, 7 cols
        
        # Conv1: Expands features. looks for small local patterns
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(64) # Normalization helps faster convergence
        
        # Conv2: Goes deeper
        self.conv2 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(128)
        
        # Conv3: Refines features
        self.conv3 = nn.Conv2d(in_channels=128, out_channels=128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)

        # --- Fully Connected Block ---
        # Flatten: 128 channels * 6 rows * 7 cols = 5376
        self.fc1 = nn.Linear(128 * 6 * 7, 768)
        self.fc2 = nn.Linear(768, 384)
        self.fc3 = nn.Linear(384, output_dim) # Output is 7 (one Q-value per column)

    def forward(self, x):
        # 1. Reshape Input
        # The environment gives us a flat vector (Batch, 42).
        # We must reshape it to (Batch, 1, 6, 7) for the CNN.
        x = x.view(-1, 1, 6, 7)
        
        # 2. Convolutions + Activations
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        
        # 3. Flatten
        x = x.view(x.size(0), -1)
        
        # 4. Dense Layers
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        
        # 5. Output (No activation here, raw Q-values)
        actions = self.fc3(x)
        
        return actions

In [9]:
import random
from collections import deque, namedtuple

Transition = namedtuple('Transition', ('state', 'action', 'reward', 'next_state', 'done'))

class ReplayMemory:
    def __init__(self, capacity):
        self.memory = deque([], maxlen=capacity)

    def push(self, *args):
        """Save a transition"""
        self.memory.append(Transition(*args))

    def sample(self, batch_size):
        return random.sample(self.memory, batch_size)

    def __len__(self):
        return len(self.memory)

In [ ]:
import torch.optim as optim
import math

class DQNAgent:
    def __init__(self, input_dim, output_dim):
        self.output_dim = output_dim
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        # 1. Initialize Networks
        # Policy Net: The one we train
        self.policy_net = QNConnectFour(output_dim).to(self.device)
        # Target Net: A stable copy to calculate future rewards (stabilizes training)
        self.target_net = QNConnectFour(output_dim).to(self.device)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval() # Set to evaluation mode

        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=0.00001)
        self.memory = ReplayMemory(250000)

        # Hyperparameters
        self.BATCH_SIZE = 256
        self.GAMMA = 0.99  # Discount factor (cares about long term)
        self.EPS_START = 1.0
        self.episode_count = 0  # Track episodes for milestone-based epsilon

        # Custom epsilon schedule milestones
        self.EPS_MILESTONE_1 = 20000   # Episode to reach 2.5%
        self.EPS_MILESTONE_2 = 100000  # Episode to drop to 1%
        self.EPS_MID = 0.015  # 2.5%
        self.EPS_LATE = 0.01  # 1%

    def get_epsilon(self):
        """
        Custom epsilon schedule:
        - Episodes 1-20k: Exponential decay to 2.5%
        - Episodes 20k-100k: Fixed at 2.5%
        - Episodes 100k+: Fixed at 1%
        """
        if self.episode_count < self.EPS_MILESTONE_1:
            # Exponential decay from EPS_START to EPS_MID
            decay_progress = self.episode_count / self.EPS_MILESTONE_1
            return self.EPS_MID + (self.EPS_START - self.EPS_MID) * math.exp(-5 * decay_progress)
        elif self.episode_count < self.EPS_MILESTONE_2:
            # Hold at 2.5%
            return self.EPS_MID
        else:
            # Drop to 1%
            return self.EPS_LATE

    def select_action(self, state, valid_moves, override_epsilon=None):
        """
        Epsilon-Greedy strategy with invalid move masking.
        override_epsilon: If provided, use this epsilon instead of the schedule.
        """
        sample = random.random()
        
        if override_epsilon is not None:
            eps_threshold = override_epsilon
        else:
            eps_threshold = self.get_epsilon()

        # EXPLORATION: Pick random valid move
        if sample < eps_threshold:
            return torch.tensor([[random.choice(valid_moves)]], device=self.device, dtype=torch.long)
        
        # EXPLOITATION: Pick best move from Network
        with torch.no_grad():
            # Get Q-values from network
            q_values = self.policy_net(state.to(self.device))
            
            # Mask invalid moves: Set their Q-value to negative infinity so they aren't picked
            # Create a mask of -inf
            mask = torch.full_like(q_values, -float('inf'))
            # Set valid indices to the actual q_values
            mask[0, valid_moves] = q_values[0, valid_moves]
            
            # Return index of max value
            return mask.max(1)[1].view(1, 1)

    def optimize_model(self):
        if len(self.memory) < self.BATCH_SIZE:
            return

        transitions = self.memory.sample(self.BATCH_SIZE)
        batch = Transition(*zip(*transitions))

        # Convert batch data to tensors
        state_batch = torch.cat(batch.state).to(self.device)
        action_batch = torch.cat(batch.action).to(self.device)
        reward_batch = torch.cat(batch.reward).to(self.device)
        next_state_batch = torch.cat(batch.next_state).to(self.device)
        done_batch = torch.cat(batch.done).to(self.device)

        # 1. Compute Q(s_t, a) - The Q-values we estimated
        state_action_values = self.policy_net(state_batch).gather(1, action_batch)

        # 2. Compute V(s_{t+1}) for all next states using Target Net
        next_state_values = self.target_net(next_state_batch).max(1)[0].detach()
        
        # 3. Compute the expected Q values (Bellman Equation)
        # If done, expected_q is just reward. If not, reward + gamma * best_future_q
        expected_state_action_values = reward_batch + (self.GAMMA * next_state_values * (1 - done_batch))

        # 4. Compute Huber Loss (Smooth L1)
        criterion = nn.SmoothL1Loss()
        loss = criterion(state_action_values, expected_state_action_values.unsqueeze(1))

        # 5. Optimize the model
        self.optimizer.zero_grad()
        loss.backward()
        
        # Clip gradients to prevent exploding gradients (common in RL)
        torch.nn.utils.clip_grad_value_(self.policy_net.parameters(), 100)
        self.optimizer.step()

        return loss

## **Enhanced Training loop to iterate over "updates"**

In [ ]:
import itertools
import copy
import math
import torch

def training_enhanced(env, agent, warmup_episodes=3000, win_rate_threshold=0.85, 
                      convergence_limit=5000, model_save_path="connect4_dqn_optimal.pth"):
    """
    Enhanced training with:
    1. Warmup phase (3000 episodes vs random opponent)
    2. Self-play with promotions (85% win rate)
    3. Custom epsilon schedule (decay to 2.5% by 20k, hold until 100k, then 1%)
    4. Convergence check (stop if no promotion after 5000 episodes)
    5. Keeps track of BEST model (highest win rate) in current promotion
    6. Final optimization phase (epsilon=0%) starting from BEST model
    7. Single-path saving (overwrites the same file on each promotion)
    """
    
    print("="*60)
    print("PHASE 1: WARMUP - Training vs Random Opponent")
    print("="*60)
    
    # Phase 1: Warmup against random opponent
    opponent_net = None  # None means random opponent
    win_history = []
    loss_history = []
    
    for i_episode in range(1, warmup_episodes + 1):
        agent.episode_count = i_episode
        
        state_np = env.reset()
        state = torch.tensor(state_np, dtype=torch.float32).unsqueeze(0)
        
        total_reward = 0
        done = False
        
        episode_loss_sum = 0
        episode_opt_count = 0
        
        while not done:
            valid_moves = env.available_actions_idx()
            action_tensor = agent.select_action(state, valid_moves)
            action = action_tensor.item()
            
            next_state_np, reward, done, info = env.step(action, opponent_model=opponent_net)
            
            reward_tensor = torch.tensor([reward], dtype=torch.float32)
            next_state = torch.tensor(next_state_np, dtype=torch.float32).unsqueeze(0)
            done_tensor = torch.tensor([float(done)], dtype=torch.float32)
            
            agent.memory.push(state, action_tensor, reward_tensor, next_state, done_tensor)
            
            state = next_state
            total_reward += reward
            
            loss = agent.optimize_model()
            if loss is not None:
                episode_loss_sum += loss.item()
                episode_opt_count += 1
        
        # Track metrics
        if episode_opt_count > 0:
            loss_history.append(episode_loss_sum / episode_opt_count)
        else:
            loss_history.append(0)
            
        if info['result'] == 'Win':
            win_history.append(1)
        else:
            win_history.append(0)
        
        if len(win_history) > 100: win_history.pop(0)
        if len(loss_history) > 100: loss_history.pop(0)
        
        # Progress report
        if i_episode % 500 == 0:
            win_rate = sum(win_history) / len(win_history) if win_history else 0
            avg_loss = sum(loss_history) / len(loss_history) if loss_history else 0
            eps = agent.get_epsilon()
            print(f"Warmup Episode {i_episode}/{warmup_episodes} | Win Rate: {win_rate:.2f} | Loss: {avg_loss:.6f} | Eps: {eps:.4f}")
    
    warmup_win_rate = sum(win_history) / len(win_history) if win_history else 0
    print(f"\n✓ Warmup Complete! Final Win Rate vs Random: {warmup_win_rate:.2f}")
    
    # Initialize opponent as copy of trained warmup agent
    opponent_net = copy.deepcopy(agent.policy_net)
    opponent_net.eval()
    
    # Update target network to match policy network after warmup
    agent.target_net.load_state_dict(agent.policy_net.state_dict())
    
    print("\n" + "="*60)
    print("PHASE 2: SELF-PLAY - Training vs Self with Promotions")
    print("="*60)
    
    # Phase 2: Self-play with promotions
    win_history = []
    loss_history = []
    promotions_count = 0
    episodes_since_last_promotion = 0
    
    # NEW: Track the best model in the current promotion cycle
    best_win_rate_in_promotion = 0.0
    best_model_state = None
    
    for i_episode in itertools.count(start=warmup_episodes + 1):
        agent.episode_count = i_episode
        episodes_since_last_promotion += 1
        
        # Convergence check
        if episodes_since_last_promotion >= convergence_limit:
            print(f"\n🎯 CONVERGENCE! No promotion after {convergence_limit} episodes.")
            print(f"Total promotions achieved: {promotions_count}")
            
            # NEW: Restore the best model from this promotion cycle
            if best_model_state is not None:
                print(f"📦 Restoring best model from this promotion (Win Rate: {best_win_rate_in_promotion:.2f})")
                agent.policy_net.load_state_dict(best_model_state)
                agent.target_net.load_state_dict(best_model_state)
                # Save this best model
                torch.save(best_model_state, model_save_path)
                print(f"💾 Best model saved to: {model_save_path}")
            break
        
        state_np = env.reset()
        state = torch.tensor(state_np, dtype=torch.float32).unsqueeze(0)
        
        total_reward = 0
        done = False
        
        episode_loss_sum = 0
        episode_opt_count = 0
        
        while not done:
            valid_moves = env.available_actions_idx()
            action_tensor = agent.select_action(state, valid_moves)
            action = action_tensor.item()
            
            next_state_np, reward, done, info = env.step(action, opponent_model=opponent_net)
            
            reward_tensor = torch.tensor([reward], dtype=torch.float32)
            next_state = torch.tensor(next_state_np, dtype=torch.float32).unsqueeze(0)
            done_tensor = torch.tensor([float(done)], dtype=torch.float32)
            
            agent.memory.push(state, action_tensor, reward_tensor, next_state, done_tensor)
            
            state = next_state
            total_reward += reward
            
            loss = agent.optimize_model()
            if loss is not None:
                episode_loss_sum += loss.item()
                episode_opt_count += 1
        
        # Track metrics
        if episode_opt_count > 0:
            loss_history.append(episode_loss_sum / episode_opt_count)
        else:
            loss_history.append(0)
            
        if info['result'] == 'Win':
            win_history.append(1)
        else:
            win_history.append(0)
        
        if len(win_history) > 100: win_history.pop(0)
        if len(loss_history) > 100: loss_history.pop(0)
        
        # Check for promotion
        if i_episode % 250 == 0 and len(win_history) == 100:
            win_rate = sum(win_history) / 100
            avg_loss = sum(loss_history) / len(loss_history)
            eps = agent.get_epsilon()
            
            # NEW: Track the best model within this promotion cycle
            if win_rate > best_win_rate_in_promotion:
                best_win_rate_in_promotion = win_rate
                best_model_state = copy.deepcopy(agent.policy_net.state_dict())
                print(f"New best in promotion! Win Rate: {win_rate:.2f}")
            
            print(f"Episode {i_episode} | Win Rate: {win_rate:.2f} | Loss: {avg_loss:.6f} | Eps: {eps:.4f} | Promotions: {promotions_count} | Since Last: {episodes_since_last_promotion}")
            
            if win_rate > win_rate_threshold:
                promotions_count += 1
                episodes_since_last_promotion = 0
                print(f"🚀 PROMOTION #{promotions_count}! Agent beats opponent at {win_rate:.2f} win rate.")
                
                # Update opponent
                opponent_net = copy.deepcopy(agent.policy_net)
                opponent_net.eval()
                
                # Update target network
                agent.target_net.load_state_dict(agent.policy_net.state_dict())
                
                # Save to single path (overwrites previous)
                torch.save(agent.policy_net.state_dict(), model_save_path)
                print(f"💾 Model saved to: {model_save_path}")
                
                # Reset stats and best tracking for NEW promotion cycle
                win_history = []
                loss_history = []
                best_win_rate_in_promotion = 0.0
                best_model_state = None
    
    # Update opponent to be the best model
    opponent_net = copy.deepcopy(agent.policy_net)
    opponent_net.eval()
    
    print("\n" + "="*60)
    print("PHASE 3: FINAL OPTIMIZATION - Pure Exploitation (ε=0%)")
    print("="*60)
    
    # Phase 3: Final optimization with epsilon=0
    final_episodes = 20000
    win_history = []
    
    # NEW: Track best model in Phase 3 as well
    best_final_win_rate = 0.0
    best_final_model_state = copy.deepcopy(agent.policy_net.state_dict())
    
    for i_episode in range(1, final_episodes + 1):
        state_np = env.reset()
        state = torch.tensor(state_np, dtype=torch.float32).unsqueeze(0)
        
        done = False
        
        while not done:
            valid_moves = env.available_actions_idx()
            # Force epsilon=0 for pure exploitation
            action_tensor = agent.select_action(state, valid_moves, override_epsilon=0.0)
            action = action_tensor.item()
            
            next_state_np, reward, done, info = env.step(action, opponent_model=opponent_net)
            
            reward_tensor = torch.tensor([reward], dtype=torch.float32)
            next_state = torch.tensor(next_state_np, dtype=torch.float32).unsqueeze(0)
            done_tensor = torch.tensor([float(done)], dtype=torch.float32)
            
            agent.memory.push(state, action_tensor, reward_tensor, next_state, done_tensor)
            
            state = next_state
            
            agent.optimize_model()
        
        if info['result'] == 'Win':
            win_history.append(1)
        else:
            win_history.append(0)
        
        if len(win_history) > 100: win_history.pop(0)
        
        if i_episode % 200 == 0:
            current_win_rate = sum(win_history) / len(win_history) if win_history else 0
            
            # NEW: Track best model during final optimization
            if current_win_rate > best_final_win_rate:
                best_final_win_rate = current_win_rate
                best_final_model_state = copy.deepcopy(agent.policy_net.state_dict())
                print(f"New best in final optimization! Win Rate: {current_win_rate:.2f}")
            
            print(f"Final Optimization Episode {i_episode}/{final_episodes} | Win Rate: {current_win_rate:.2f}")
    
    final_win_rate = sum(win_history) / len(win_history) if win_history else 0
    print(f"\n✓ Final Optimization Complete! Final Win Rate: {final_win_rate:.2f}")
    print(f"✓ Best Win Rate during Phase 3: {best_final_win_rate:.2f}")
    
    # NEW: Only save if Phase 3 improved the model
    if best_final_win_rate > best_win_rate_in_promotion:
        agent.policy_net.load_state_dict(best_final_model_state)
        torch.save(best_final_model_state, model_save_path)
        print(f"💾 Phase 3 improved the model! Saved best Phase 3 model (WR: {best_final_win_rate:.2f})")
    else:
        print(f"ℹ️  Phase 3 did not improve model. Keeping Phase 2 best model (WR: {best_win_rate_in_promotion:.2f})")
    
    print("="*60)
    print(f"TRAINING COMPLETE!")
    print(f"Total Episodes: {agent.episode_count + final_episodes}")
    print(f"Total Promotions: {promotions_count}")
    print(f"Final Model Path: {model_save_path}")
    print("="*60)
    
    return agent

In [12]:
# ============================================================
# TRAINING - Training Enhancement
# ============================================================

# 1. Initialize Environment and Agent
env = ConnectFourEnv()
agent = DQNAgent(input_dim=42, output_dim=7)

print(f"Training on device: {agent.device}")
print(f"Starting fresh training with enhanced algorithm...\n")

# 2. Run the Enhanced Training
# This will:
# - Train 5000 episodes vs random opponent (warmup)
# - Self-play with 85% win rate promotions
# - Use custom epsilon schedule (decay to 2.5% by 20k, hold until 100k, then 1%)
# - Stop if no promotion after 5000 episodes (convergence)
# - Final optimization with epsilon=0%
# - Save to single path (overwrites on each promotion)

model = training_enhanced(
    env=env,
    agent=agent,
    warmup_episodes=5000,
    win_rate_threshold=0.85,
    convergence_limit=15000,
    model_save_path="connect4_dqn_optimal.pth"
)

print("\nTraining finished! Model is ready to play.")

Training on device: cpu
Starting fresh training with enhanced algorithm...

PHASE 1: WARMUP - Training vs Random Opponent


KeyboardInterrupt: 